In [16]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [17]:
import time
from trial_timing import reset_trial_times, append_trial_time

import os
import json
import numpy as np
from config import Config

def evenly_binary_sequence(count: int, n: int) -> np.ndarray:
\
\
       
    count = int(np.clip(count, 0, n))
    seq = np.zeros(n, dtype=np.int32)
    if count == 0:
        return seq
    if count == n:
        seq[:] = 1
        return seq

    acc = 0
    for i in range(n):
        acc += count
        if acc >= n:
            seq[i] = 1
            acc -= n
    return seq

def counts_to_binary_action_plan(
    counts_block: np.ndarray,
    num_od: int,
    total_step: int,
    block_steps: int
) -> np.ndarray:
\
\
\
       
    n_blocks = int(np.ceil(total_step / block_steps))
    action_plan = np.zeros((total_step, num_od), dtype=np.int32)

    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        for od in range(num_od):
            c = int(np.rint(counts_block[b, od]))
            c = int(np.clip(c, 0, n_local))
            action_plan[s:e, od] = evenly_binary_sequence(c, n_local)

    return action_plan

def run_episode_with_plan(build_env_fn, worker_idx, action_plan, seed):
    env, obs, info = build_env_fn(
        worker_idx=worker_idx,
        total_step=action_plan.shape[0],
        seed=seed
    )
    total_reward = 0.0
    last_info = {}

    try:
        for t in range(action_plan.shape[0]):
            obs, reward, terminated, truncated, info = env.step(action_plan[t])
            total_reward += float(reward)
            last_info = info
            if terminated or truncated:
                break
    finally:
        env.close()

    trajectory = last_info.get("trajectory", [])
    return float(total_reward), trajectory

def cmaes_optimize_counts_300s(
    build_env_fn,
    worker_idx,
    num_od,
    total_step,
    input_interval,
    detector_interval,
    init_points=0,
    n_iter=200,
    seed=42,
    verbose=2,
    popsize=16,
    sigma0=12.0,
):
    block_steps = detector_interval // input_interval
    n_blocks = int(np.ceil(total_step / block_steps))

    var_meta = []
    lb = []
    ub = []
    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        for od in range(num_od):
            var_meta.append((b, od, n_local))
            lb.append(0.0)
            ub.append(float(n_local))

    lb = np.array(lb, dtype=np.float64)
    ub = np.array(ub, dtype=np.float64)
    n = lb.size

    rng = np.random.default_rng(seed)
    eval_budget = int(max(1, init_points + n_iter))
    eval_count = 0

    def x_to_counts(x_vec: np.ndarray) -> np.ndarray:
        x_vec = np.asarray(x_vec, dtype=np.float64)
        counts = np.zeros((n_blocks, num_od), dtype=np.float64)
        for i, (b, od, n_local) in enumerate(var_meta):
            counts[b, od] = float(np.clip(x_vec[i], 0.0, float(n_local)))
        return counts

    def evaluate_x(x_vec: np.ndarray):
        nonlocal eval_count
        x_vec = np.clip(np.asarray(x_vec, dtype=np.float64), lb, ub)
        counts_block = x_to_counts(x_vec)
        action_plan = counts_to_binary_action_plan(
            counts_block=counts_block,
            num_od=num_od,
            total_step=total_step,
            block_steps=block_steps
        )
        score, _ = run_episode_with_plan(
            build_env_fn=build_env_fn,
            worker_idx=worker_idx,
            action_plan=action_plan,
            seed=seed + eval_count,
        )
        eval_count += 1
        return float(score)

    lam = int(max(4, popsize))
    mu = lam // 2

    w = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    w = w / np.sum(w)
    mueff = 1.0 / np.sum(w ** 2)

    cc = (4 + mueff / n) / (n + 4 + 2 * mueff / n)
    cs = (mueff + 2) / (n + mueff + 5)
    c1 = 2.0 / ((n + 1.3) ** 2 + mueff)
    cmu = min(1.0 - c1, 2.0 * (mueff - 2.0 + 1.0 / mueff) / ((n + 2.0) ** 2 + mueff))
    damps = 1.0 + 2.0 * max(0.0, np.sqrt((mueff - 1.0) / (n + 1.0)) - 1.0) + cs
    chi_n = np.sqrt(n) * (1.0 - 1.0 / (4.0 * n) + 1.0 / (21.0 * n * n))

    m = 0.5 * (lb + ub)
    sigma = float(sigma0)
    C = np.eye(n, dtype=np.float64)
    pc = np.zeros(n, dtype=np.float64)
    ps = np.zeros(n, dtype=np.float64)

    B = np.eye(n, dtype=np.float64)
    D = np.ones(n, dtype=np.float64)
    inv_sqrt_C = np.eye(n, dtype=np.float64)

    best_score = -np.inf
    best_x = m.copy()
    history = []
    gen = 0

    def _update_eigendecomp(C_mat):

        C_sym = 0.5 * (C_mat + C_mat.T)
        eigvals, eigvecs = np.linalg.eigh(C_sym)
        eigvals = np.clip(eigvals, 1e-20, None)
        D_new = np.sqrt(eigvals)
        B_new = eigvecs
        inv_sqrt_C_new = B_new @ np.diag(1.0 / D_new) @ B_new.T
        return C_sym, B_new, D_new, inv_sqrt_C_new

    C, B, D, inv_sqrt_C = _update_eigendecomp(C)

    while eval_count + lam <= eval_budget:
        gen += 1

        arz = rng.standard_normal((lam, n))
        BD = B * D
        ary = arz @ BD.T
        arx = m[None, :] + sigma * ary
        arx = np.clip(arx, lb[None, :], ub[None, :])

        fitness = np.empty(lam, dtype=np.float64)
        for i in range(lam):
            fitness[i] = evaluate_x(arx[i])
            if fitness[i] > best_score:
                best_score = float(fitness[i])
                best_x = arx[i].copy()

        idx = np.argsort(-fitness)
        x_sel = arx[idx[:mu]]
        y_sel = ary[idx[:mu]]
        z_sel = arz[idx[:mu]]

        y_w = np.sum(w[:, None] * y_sel, axis=0)
        z_w = np.sum(w[:, None] * z_sel, axis=0)
        m = m + sigma * y_w
        m = np.clip(m, lb, ub)

        ps = (1 - cs) * ps + np.sqrt(cs * (2 - cs) * mueff) * (inv_sqrt_C @ y_w)
        norm_ps = np.linalg.norm(ps)

        hsig_lhs = norm_ps / np.sqrt(1 - (1 - cs) ** (2 * gen)) / chi_n
        hsig = 1.0 if hsig_lhs < (1.4 + 2.0 / (n + 1.0)) else 0.0

        pc = (1 - cc) * pc + hsig * np.sqrt(cc * (2 - cc) * mueff) * y_w

        rank_one = np.outer(pc, pc)
        rank_mu = (y_sel * w[:, None]).T @ y_sel
        C = (1 - c1 - cmu) * C + c1 * (rank_one + (1 - hsig) * cc * (2 - cc) * C) + cmu * rank_mu

        sigma = sigma * np.exp((cs / damps) * (norm_ps / chi_n - 1.0))
        sigma = float(np.clip(sigma, 1e-6, 50.0))

        if gen == 1 or gen % max(1, n // 10) == 0:
            C, B, D, inv_sqrt_C = _update_eigendecomp(C)

        history.append({
            "gen": int(gen),
            "eval": int(eval_count),
            "best_score": float(best_score),
            "mean_fitness": float(np.mean(fitness)),
            "sigma": float(sigma),
        })

        if verbose >= 2 and (gen == 1 or gen % 5 == 0):
            print(f"[CMA-ES] gen {gen} | eval {eval_count}/{eval_budget} | best={best_score:.6f} | sigma={sigma:.4f}")

    while eval_count < eval_budget:
        xr = best_x + rng.normal(0.0, 1.0, size=n) * (0.2 * sigma + 1.0)
        xr = np.clip(xr, lb, ub)
        yr = evaluate_x(xr)
        history.append({"eval": int(eval_count), "phase": "tail_local", "score": float(yr)})
        if yr > best_score:
            best_score = float(yr)
            best_x = xr.copy()

    best_counts = x_to_counts(best_x)
    best_actions = counts_to_binary_action_plan(
        counts_block=best_counts,
        num_od=num_od,
        total_step=total_step,
        block_steps=block_steps
    )

    cma_log = {
        "method": "CMA-ES",
        "dim": int(n),
        "eval_budget": int(eval_budget),
        "total_evals": int(eval_count),
        "popsize": int(lam),
        "mu": int(mu),
        "sigma0": float(sigma0),
        "sigma_final": float(sigma),
        "generations": int(gen),
        "cc": float(cc),
        "cs": float(cs),
        "c1": float(c1),
        "cmu": float(cmu),
        "damps": float(damps),
        "history": history,
    }

    return best_actions, float(best_score), cma_log, best_counts

os.makedirs(Config.RESULT_DIR, exist_ok=True)

_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
reset_trial_times(_trial_result_dir)

for trial in range(5):
    _trial_t0 = time.perf_counter()
    trial_idx = trial
    seed_opt = int(100 + (trial_idx + 1))
                                                                          
    seed_eval = trial

    best_actions, best_score, cma_log, best_counts = cmaes_optimize_counts_300s(
        build_env_fn=build_env,
        worker_idx=0,
        num_od=Config.NUM_OD,
        total_step=Config.TOTAL_STEP,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        init_points=0,
        n_iter=600,
        seed=seed_opt,
        verbose=2,
        popsize=16,
        sigma0=12.0,
    )

    print(f"[trial {trial}] Best total_reward:", best_score)
    print(f"[trial {trial}] Best action shape:", best_actions.shape)
    print(f"[trial {trial}] Best counts shape:", best_counts.shape)

    replay_reward, trajectory = run_episode_with_plan(
        build_env_fn=build_env,
        worker_idx=0,
        action_plan=best_actions,
        seed=seed_eval
    )
    print(f"[trial {trial}] Replay reward:", replay_reward)

    with open(os.path.join(Config.RESULT_DIR, f"trajectory_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(
            trajectory, f, ensure_ascii=False, indent=2,
            default=lambda o: o.tolist() if isinstance(o, np.ndarray) else o
        )

    with open(os.path.join(Config.RESULT_DIR, f"cma_log_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(cma_log, f, ensure_ascii=False, indent=2)
    append_trial_time(_trial_result_dir, trial, time.perf_counter() - _trial_t0)


ans: [11.  0.  3.  4.  5.  1. 20. 21.  4.], det: [26.  0. 26. 13.  4.  0. 18. 13. 11.], reward: -106.0
ans: [ 8. 11.  3. 14. 17.  9. 19. 30.  3.], det: [28. 20.  7. 33. 15. 19. 33. 56. 12.], reward: -212.77777099609375
ans: [23.  4. 13. 20. 17. 11. 15. 23.  5.], det: [27. 19. 17. 33. 39. 10. 36. 63. 12.], reward: -333.4444580078125
ans: [15. 10. 10. 13. 17.  5. 11. 26.  2.], det: [33. 41.  8. 26. 26. 13. 39. 73. 13.], reward: -524.111083984375
ans: [19.  4.  8. 21. 19. 14. 14. 33.  7.], det: [50. 16. 52. 62. 26. 34. 39. 54. 17.], reward: -704.111083984375
ans: [14. 15. 15. 14. 12.  7. 13. 23.  6.], det: [40. 29. 17. 31. 70. 29. 28. 76.  6.], reward: -894.111083984375
Environment closed.
ans: [11.  0.  3.  4.  5.  1. 20. 21.  4.], det: [32.  0. 13. 10.  9.  0. 51. 34. 10.], reward: -195.55555725097656
ans: [ 8. 11.  3. 14. 17.  9. 19. 30.  3.], det: [36. 22. 12. 40. 34. 20. 63. 77. 13.], reward: -701.888916015625
ans: [23.  4. 13. 20. 17. 11. 15. 23.  5.], det: [11. 31.  7. 41. 26. 14. 

In [18]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
